# 01 — Train Medical SAM Adapter (Phase A, Stage 2)

Train lesion-level adapter di DENTEX (Wu et al., 2025). SAM ViT-H frozen, hanya adapter + mask decoder dilatih (~2-3%). Checkpoint → Drive.

**Prasyarat:** `00_setup.ipynb` sudah sukses (Drive mount, SAM ViT-H di Drive, DENTEX verified).

Runtime → **T4/L4 GPU** → Run all. Training ~beberapa jam (ViT-H berat).

## Cell 1 — Mount Drive + clone/pull repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}/scripts

## Cell 2 — Install deps (kalau runtime baru)

In [ ]:
%pip install -q segment-anything pycocotools opencv-python-headless
import torch; print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Smoke test dataset (harus 3529 lesion, 1 sample valid)

In [ ]:
import os
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
# path absolut -> tidak tergantung folder aktif
!python /content/opg-live/scripts/dentex_dataset.py

## Cell 4 — Cek trainable params (~2-3%) sebelum training
Sanity check adapter ke-inject benar & base frozen.

In [ ]:
import torch
from segment_anything import sam_model_registry
from sam_adapter import inject_adapters, trainable_report

sam = sam_model_registry['vit_h'](checkpoint=f'{DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth')
inject_adapters(sam)
trainable_report(sam)   # target: beberapa % dari 641M

## Cell 5 — Training
Mulai pendek dulu (`--epochs 2`) untuk pastikan loop jalan & loss turun. Kalau OK, naikkan epoch. Checkpoint terbaik otomatis ke `Drive/checkpoints/adapter_best.pth`.

In [ ]:
!python /content/opg-live/scripts/train_adapter.py \
  --drive {DRIVE_ROOT} \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 2 --bs 2 --accum 4 --lr 1e-4

## Cell 6 — Full training (setelah Cell 5 terbukti loss turun)
Naikkan epoch. **Checkpoint ke Drive = aman dari reset Colab.**

In [ ]:
!python /content/opg-live/scripts/train_adapter.py \
  --drive {DRIVE_ROOT} \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 15 --bs 2 --accum 4 --lr 1e-4

## ✅ Deliverable Phase A (Stage 2)
- [ ] `adapter_best.pth` di Drive (~20-50 MB)
- [ ] val Dice global terlaporkan
- [ ] **Dice per-kelas** terlaporkan (Impacted/Caries/Periapical/Deep Caries) — penting untuk H4
- [ ] Target Dice ≥ 0.80 (lesion-level)

**Catatan H4:** Periapical n=158 → Dice-nya kemungkinan paling rendah & varian tinggi. Itu wajar, dilaporkan apa adanya (effect-size primary).

**Next:** `02_auto_pipeline.ipynb` (Stage 1 HierarchicalDet + Stage 2 adapter → 3-arm artifact: bbox / mask / hybrid).